# 05 Tree Baselines (Local-only + Centralized)

This notebook runs reviewer-requested tree baselines for direct comparison with MLP/LSTM:
- Local-only XGBoost
- Centralized global XGBoost
- Local-only LightGBM
- Centralized global LightGBM

Outputs are saved to `saved_models/` as CSV files.

In [ ]:
# Optional install (run once if needed)
# !pip install xgboost lightgbm

import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 2024
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path('ashrae')
SAVE_DIR = Path('saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.cleaned_isamu_matt.building_mean_y_ge_1.csv'
WEATHER_PATH = DATA_DIR / 'weather_train.cleaned.csv'
META_PATH = DATA_DIR / 'building_metadata.csv'

assert TRAIN_PATH.exists(), f'Missing file: {TRAIN_PATH}'
assert WEATHER_PATH.exists(), f'Missing file: {WEATHER_PATH}'
assert META_PATH.exists(), f'Missing file: {META_PATH}'


In [2]:
# Load data
train_raw = pd.read_csv(TRAIN_PATH)
weather = pd.read_csv(WEATHER_PATH)
meta = pd.read_csv(META_PATH)

for d in (train_raw, weather):
    if 'timestamp' in d.columns:
        d['timestamp'] = pd.to_datetime(d['timestamp'])

# Merge metadata without duplicating columns already present in cleaned train
meta_extra_cols = [c for c in meta.columns if c not in train_raw.columns]
if meta_extra_cols:
    df = train_raw.merge(meta[['building_id'] + meta_extra_cols], on='building_id', how='left')
else:
    df = train_raw.copy()

if 'site_id' not in df.columns:
    raise KeyError('site_id is missing after metadata merge. Check train_raw/meta columns.')
df = df.merge(weather, on=['site_id', 'timestamp'], how='left')

# Keep electric meter only when meter exists
if 'meter' in df.columns:
    df = df[df['meter'] == 0].copy()

# Encode primary_use
if 'primary_use' in df.columns:
    df['primary_use_enc'] = pd.factorize(df['primary_use'])[0]
else:
    df['primary_use_enc'] = 0

# Ensure target
if 'meter_reading' not in df.columns:
    raise RuntimeError('Expected meter_reading column is missing.')

df['meter_reading'] = df['meter_reading'].clip(lower=0)
df['y'] = np.log1p(df['meter_reading'])

print('Rows after merge/filter:', len(df))
print('Buildings:', df['building_id'].nunique())

Rows after merge/filter: 11362529
Buildings: 1377


In [3]:
# Feature engineering aligned with MLP/LSTM setup
df['hour'] = df['timestamp'].dt.hour
df['dow'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['log_sqft'] = np.log1p(df['square_feet'].fillna(df['square_feet'].median()))
df['year_built'] = df['year_built'].fillna(df['year_built'].median())
df['building_age'] = 2017 - df['year_built']
df['floor_count'] = df['floor_count'].fillna(1)

weather_cols = [
    'air_temperature', 'dew_temperature', 'wind_speed', 'cloud_coverage',
    'precip_depth_1_hr', 'sea_level_pressure'
]
for c in weather_cols:
    if c not in df.columns:
        df[c] = 0.0
    df[c] = df[c].fillna(df[c].median())

# Lags per building
df = df.sort_values(['building_id', 'timestamp']).copy()
g = df.groupby('building_id')['y']
df['lag_1'] = g.shift(1)
df['lag_24'] = g.shift(24)
df['roll24_mean'] = g.shift(1).rolling(24).mean().reset_index(level=0, drop=True)
df['roll24_std'] = g.shift(1).rolling(24).std().reset_index(level=0, drop=True)

for c in ['lag_1', 'lag_24', 'roll24_mean', 'roll24_std']:
    df[c] = df[c].fillna(df[c].median())

FEATURES = [
    'site_id', 'building_id', 'primary_use_enc',
    'hour', 'dow', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'log_sqft', 'building_age', 'floor_count',
    'lag_1', 'lag_24', 'roll24_mean', 'roll24_std',
] + weather_cols

TARGET = 'y'

print('Features:', len(FEATURES))

Features: 26


In [4]:
# Build per-building train/val/test client split
TRAIN_FRAC = 0.75
VAL_FRAC = 0.10
MIN_ROWS_PER_BUILDING = 2000
N_CLIENTS = 60
NON_IID_BY_SITE = True
SITES_TO_USE = 6

all_buildings = df['building_id'].unique()
if NON_IID_BY_SITE:
    site_counts = df.groupby('site_id')['building_id'].nunique().sort_values(ascending=False)
    top_sites = site_counts.head(SITES_TO_USE).index.tolist()
    candidate_buildings = df[df['site_id'].isin(top_sites)]['building_id'].unique()
    chosen_buildings = np.random.choice(candidate_buildings, size=min(N_CLIENTS, len(candidate_buildings)), replace=False)
else:
    chosen_buildings = np.random.choice(all_buildings, size=min(N_CLIENTS, len(all_buildings)), replace=False)

chosen_buildings = sorted(chosen_buildings.tolist())
client_data = {}

def time_split_building(d):
    d = d.sort_values('timestamp').reset_index(drop=True)
    n = len(d)
    n_train = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    tr = d.iloc[:n_train].copy()
    va = d.iloc[n_train:n_train + n_val].copy()
    te = d.iloc[n_train + n_val:].copy()
    return tr, va, te

for bid in chosen_buildings:
    d = df[df['building_id'] == bid].copy()
    if len(d) < MIN_ROWS_PER_BUILDING:
        continue
    tr, va, te = time_split_building(d)
    if min(len(tr), len(va), len(te)) < 50:
        continue
    client_data[bid] = {'train': tr, 'val': va, 'test': te}

usable_bids = sorted(client_data.keys())
print('Usable clients:', len(usable_bids))

Usable clients: 58


In [5]:
# Cold-start setup + save helper
ENABLE_COLD_START = True
COLD_START_RATIO = 0.20
COLD_TRAIN_DAYS = 14
MIN_TRAIN_ROWS_COLD = 300

if ENABLE_COLD_START and len(usable_bids) > 0:
    cold_set = set(np.random.choice(usable_bids, size=max(1, int(COLD_START_RATIO * len(usable_bids))), replace=False))
    for bid in cold_set:
        tr = client_data[bid]['train'].copy()
        start = tr['timestamp'].min()
        tr_cs = tr[tr['timestamp'] < (start + pd.Timedelta(days=COLD_TRAIN_DAYS))].copy()
        if len(tr_cs) < MIN_TRAIN_ROWS_COLD:
            tr_cs = tr.iloc[:MIN_TRAIN_ROWS_COLD].copy()
        client_data[bid]['train_orig'] = tr.copy()
        client_data[bid]['train'] = tr_cs
else:
    cold_set = set()

source_bids = [b for b in usable_bids if b not in cold_set]

def save_dataframe(df, name):
    out = SAVE_DIR / f'{name}.csv'
    df.to_csv(out, index=False)
    print(f'Saved: {out}')

print('Cold-start clients:', len(cold_set))
print('Warm/source clients:', len(source_bids))

Cold-start clients: 11
Warm/source clients: 47


In [6]:
# Run reviewer-requested tree baselines
import importlib
import run_tree_baselines as tree_baselines

tree_baselines = importlib.reload(tree_baselines)
run_tree_baselines = tree_baselines.run_tree_baselines

tree_results = run_tree_baselines(
    seed=SEED,
    client_data=client_data,
    usable_bids=usable_bids,
    target=TARGET,
    features=FEATURES,
    cold_set=cold_set,
    source_bids=source_bids,
)
globals().update(tree_results)

# Save seed-prefixed copies in saved_models for easy tracking per run
if 'df_central_xgb' in globals():
    save_dataframe(df_central_xgb, f'seed{SEED}_tree_centralized_xgb')
if 'df_local_xgb' in globals():
    save_dataframe(df_local_xgb, f'seed{SEED}_tree_local_only_xgb')
if 'df_central_lgbm' in globals():
    save_dataframe(df_central_lgbm, f'seed{SEED}_tree_centralized_lgbm')
if 'df_local_lgbm' in globals():
    save_dataframe(df_local_lgbm, f'seed{SEED}_tree_local_only_lgbm')

print('\nSummary table:')
df_tree_baseline_summary




Tree Baselines - XGBoost
Saved: saved_results\model_tree_centralized_xgb_seed2024.csv
Saved: saved_results\model_tree_local_only_xgb_seed2024.csv

Centralized XGBoost
  Mean MAE (log1p): 0.1028
  Mean RMSE (log1p): 0.2299
  Mean CVRMSE (raw %): 12.95%
  Mean WAPE (raw %): 8.06%
  Cold-start MAE: 0.1787
  Cold-start CVRMSE: 23.49%

Local-only XGBoost
  Mean MAE (log1p): 0.1205
  Mean RMSE (log1p): 0.2567
  Mean CVRMSE (raw %): 14.32%
  Mean WAPE (raw %): 9.40%
  Cold-start MAE: 0.1735
  Cold-start CVRMSE: 19.85%

Tree Baselines - LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005129 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2341
[LightGBM] [Info] Number of data points in the train set: 299758, number of used features: 25
[LightGBM] [Info] Start training from score 4.615832
[LightGBM] [Info] Auto-choosing col-wise multi-th

,Model,Seed,Strategy,Overall,Warm,Cold,CVRMSE (%)
2,MODEL,2024,C. Centralized-LightGBM,0.081389,0.074830,0.109413,9.786383
0,MODEL,2024,C. Centralized-XGBoost,0.102809,0.085045,0.178707,12.946930
3,MODEL,2024,0. Local-only-LightGBM,0.106502,0.092519,0.166246,12.915049
1,MODEL,2024,0. Local-only-XGBoost,0.120536,0.108130,0.173544,14.319285


In [1]:
# Compare Local-only vs Centralized baselines for seeds 42, 123, 2024 only

from pathlib import Path
import re
import numpy as np
import pandas as pd

TARGET_SEEDS = {42, 123, 2024}
METRICS = ["overall", "cold", "warm", "cv_rmse"]

METRIC_ALIASES = {
    "overall": ["overall", "mae", "overall_mae", "test_mae"],
    "cold": ["cold", "cold_mae"],
    "warm": ["warm", "warm_mae"],
    "cv_rmse": ["cv_rmse", "cvrmse", "cv-rmse", "cv rmse", "overall_cvrmse"],
}

def normalize_col(c):
    c = str(c).strip().lower().replace("%", "pct")
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

def pick_metric_column(df, metric):
    norm_map = {}
    for c in df.columns:
        k = normalize_col(c)
        if k not in norm_map:
            norm_map[k] = c
    for alias in METRIC_ALIASES[metric]:
        k = normalize_col(alias)
        if k in norm_map:
            return norm_map[k]
    return None

def summarize_from_df(df, model_label, mode, files_label):
    per_seed = {m: [] for m in METRICS}
    for _, g in df.groupby("seed"):
        for m in METRICS:
            col = pick_metric_column(g, m)
            if col is None:
                continue
            vals = pd.to_numeric(g[col], errors="coerce").dropna()
            if len(vals):
                per_seed[m].append(float(vals.mean()))

    if not any(len(v) > 0 for v in per_seed.values()):
        return None

    row = {"Model": model_label, "Mode": mode, "Seeds (n)": max(len(v) for v in per_seed.values()), "Files": files_label}
    for m in METRICS:
        vals = per_seed[m]
        if len(vals) == 0:
            row[f"{m} mean"] = np.nan
            row[f"{m} std"] = np.nan
            row[f"{m} mean +/- std"] = "N/A"
        else:
            mean_v = float(np.mean(vals))
            std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            row[f"{m} mean"] = mean_v
            row[f"{m} std"] = std_v
            row[f"{m} mean +/- std"] = f"{mean_v:.3f} +/- {std_v:.3f}"
    return row

rows = []

# A) Summary models: MLP/LSTM/PatchTST
summary_files = sorted(
    list(Path("saved_results").glob("mlp_seed_*_summary.csv")) +
    list(Path("saved_results").glob("lstm_seed_*_summary.csv")) +
    list(Path("saved_results").glob("patchtst_seed_*_summary.csv"))
)

if summary_files:
    s = pd.concat([pd.read_csv(f) for f in summary_files], ignore_index=True)
    s.columns = [normalize_col(c) for c in s.columns]
    s = s.loc[:, ~s.columns.duplicated(keep="first")]

    if {"model", "seed", "strategy"}.issubset(s.columns):
        s["seed"] = pd.to_numeric(s["seed"], errors="coerce")
        s = s[s["seed"].isin(TARGET_SEEDS)].copy()

        model_s = s["model"].astype(str).str.strip().str.lower()
        strat_s = s["strategy"].astype(str).str.strip().str.lower()

        valid = model_s.ne("nan") & model_s.ne("") & strat_s.ne("nan") & strat_s.ne("")
        s = s[valid].copy()
        model_s = model_s[valid]
        strat_s = strat_s[valid]

        def pick_neural(model_name, mode):
            m = model_s.eq(model_name.lower())
            k = strat_s.eq("0. local-only") if mode == "Local-only" else strat_s.eq("c. centralized")
            return s[m & k].copy()

        for mdl in ["MLP", "LSTM", "PatchTST"]:
            for mode in ["Local-only", "Centralized"]:
                sub = pick_neural(mdl, mode)
                if not sub.empty:
                    r = summarize_from_df(sub, mdl, mode, ", ".join(str(f) for f in summary_files))
                    if r is not None:
                        rows.append(r)

# B) Tree models from raw files
def summarize_raw(model, mode, pattern):
    files = sorted(Path("saved_models").glob(pattern))
    if not files:
        return None

    parts = []
    for f in files:
        d = pd.read_csv(f)
        if d.empty:
            continue
        d = d.copy()
        if "seed" not in d.columns:
            m = re.search(r"seed(\d+)", f.stem.lower())
            d["seed"] = int(m.group(1)) if m else np.nan
        d["seed"] = pd.to_numeric(d["seed"], errors="coerce")
        d = d[d["seed"].isin(TARGET_SEEDS)]
        if not d.empty:
            parts.append(d)

    if not parts:
        return None

    return summarize_from_df(pd.concat(parts, ignore_index=True), model, mode, ", ".join(str(f) for f in files))

for model, mode, pat in [
    ("XGBoost", "Local-only", "seed*_tree_local_only_xgb.csv"),
    ("XGBoost", "Centralized", "seed*_tree_centralized_xgb.csv"),
    ("LightGBM", "Local-only", "seed*_tree_local_only_lgbm.csv"),
    ("LightGBM", "Centralized", "seed*_tree_centralized_lgbm.csv"),
]:
    r = summarize_raw(model, mode, pat)
    if r is not None:
        rows.append(r)

compare_df = pd.DataFrame(rows)
if not compare_df.empty:
    compare_df = compare_df.sort_values(["Mode", "overall mean"], ascending=[True, True]).reset_index(drop=True)
    ncols = compare_df.select_dtypes(include=[np.number]).columns
    compare_df[ncols] = compare_df[ncols].round(3)
    compare_df["Seeds (n)"] = compare_df["Seeds (n)"].astype(int)
    for m in METRICS:
        compare_df[f"{m} mean +/- std"] = np.where(
            compare_df[f"{m} mean"].notna(),
            compare_df[f"{m} mean"].map(lambda x: f"{x:.3f}") + " +/- " + compare_df[f"{m} std"].fillna(0).map(lambda x: f"{x:.3f}"),
            "N/A"
        )

compare_df

,Model,Mode,Seeds (n),Files,overall mean,overall std,overall mean +/- std,cold mean,cold std,cold mean +/- std,warm mean,warm std,warm mean +/- std,cv_rmse mean,cv_rmse std,cv_rmse mean +/- std
0,LightGBM,Centralized,3,saved_models\seed123_tree_centralized_lgbm.csv...,0.082,0.011,0.082 +/- 0.011,NaN,NaN,N/A,NaN,NaN,N/A,10.801,1.571,10.801 +/- 1.571
1,XGBoost,Centralized,3,"saved_models\seed123_tree_centralized_xgb.csv,...",0.103,0.015,0.103 +/- 0.015,NaN,NaN,N/A,NaN,NaN,N/A,13.276,2.367,13.276 +/- 2.367
2,LSTM,Centralized,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.116,0.016,0.116 +/- 0.016,0.125,0.016,0.125 +/- 0.016,0.114,0.016,0.114 +/- 0.016,17.596,3.417,17.596 +/- 3.417
3,MLP,Centralized,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.137,0.017,0.137 +/- 0.017,0.186,0.022,0.186 +/- 0.022,0.124,0.017,0.124 +/- 0.017,16.443,1.509,16.443 +/- 1.509
4,PatchTST,Centralized,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.193,0.032,0.193 +/- 0.032,0.206,0.023,0.206 +/- 0.023,0.190,0.035,0.190 +/- 0.035,32.312,5.349,32.312 +/- 5.349
5,LightGBM,Local-only,3,"saved_models\seed123_tree_local_only_lgbm.csv,...",0.141,0.038,0.141 +/- 0.038,NaN,NaN,N/A,NaN,NaN,N/A,16.054,4.226,16.054 +/- 4.226
6,XGBoost,Local-only,3,"saved_models\seed123_tree_local_only_xgb.csv, ...",0.154,0.038,0.154 +/- 0.038,NaN,NaN,N/A,NaN,NaN,N/A,17.543,4.367,17.543 +/- 4.367
7,MLP,Local-only,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.263,0.005,0.263 +/- 0.005,0.542,0.008,0.542 +/- 0.008,0.193,0.006,0.193 +/- 0.006,28.966,0.477,28.966 +/- 0.477
8,LSTM,Local-only,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.290,0.015,0.290 +/- 0.015,0.483,0.098,0.483 +/- 0.098,0.241,0.006,0.241 +/- 0.006,29.567,0.480,29.567 +/- 0.480
9,PatchTST,Local-only,3,"saved_results\lstm_seed_123_summary.csv, saved...",0.548,0.042,0.548 +/- 0.042,1.672,0.218,1.672 +/- 0.218,0.267,0.002,0.267 +/- 0.002,35.993,0.185,35.993 +/- 0.185
